# Prompt Engineering Evaluation

## Introduction
This evaluation tested multiple prompting techniques using the Groq API. The goal was to assess how different strategies perform in realistic, ambiguous scenarios and to identify strengths, weaknesses, and improvement opportunities. The techniques covered include Zero-Shot, Few-Shot, Chain-of-Thought, LLM-as-Judge, Self-Consistency, Tree-of-Thought, and Rephrase-and-Respond.

---

## Technique-by-Technique Analysis

### Zero-Shot Prompting
- **Strengths:** Fast setup, requires minimal preparation, effective when instructions and schema are clear.  
- **Weaknesses:** Sensitive to ambiguity; may miss implicit risks or overgeneralize.  
- **Improvement:** Provide precise decision criteria and emphasize hidden risk factors.

### Few-Shot Prompting
- **Strengths:** Examples guide the model toward consistent outputs; useful for teaching boundaries.  
- **Weaknesses:** Requires careful curation of examples; risk of bias if examples are too narrow.  
- **Improvement:** Include ambiguous and edge cases to strengthen generalization.

### Chain-of-Thought Reasoning
- **Strengths:** Enables structured reasoning, particularly for numerical and diagnostic tasks.  
- **Weaknesses:** Can produce verbose or redundant reasoning; risk of incorrect math if not guided.  
- **Improvement:** Explicitly constrain reasoning to concise summaries and enforce schema validation.

### LLM-as-Judge
- **Strengths:** Effective for evaluation tasks when rubric is clear; highlights strengths and weaknesses.  
- **Weaknesses:** Susceptible to vague scoring if rubric lacks specificity.  
- **Improvement:** Define scoring rules tightly and penalize misleading or dismissive responses.

### Self-Consistency
- **Strengths:** Aggregates multiple runs to reduce variance; improves reliability in complex logic.  
- **Weaknesses:** Computationally expensive; disagreements can be difficult to resolve.  
- **Improvement:** Use majority voting and highlight disagreement analysis for transparency.

### Tree-of-Thought
- **Strengths:** Encourages exploration of alternatives and trade-offs; supports multi-criteria decisions.  
- **Weaknesses:** Can be slow and complex; risk of over-weighting one dimension.  
- **Improvement:** Balance scores across feasibility, risk, and adoption, not just business value.

### Rephrase-and-Respond
- **Strengths:** Clarifies vague or ambiguous requests; ensures solutions are practical and measurable.  
- **Weaknesses:** May over-simplify if assumptions are not carefully stated.  
- **Improvement:** Explicitly document clarified assumptions and tie them to measurable outcomes.

---

## Comparative Summary
- **Zero-Shot** is best for quick classification tasks but weak against ambiguity.  
- **Few-Shot** excels in teaching boundaries but requires careful example design.  
- **Chain-of-Thought** is powerful for reasoning but must be constrained to avoid verbosity.  
- **LLM-as-Judge** is reliable when rubric clarity is high.  
- **Self-Consistency** improves accuracy but adds computational overhead.  
- **Tree-of-Thought** is ideal for strategic decisions but requires structured scoring.  
- **Rephrase-and-Respond** is essential for ambiguous inputs, ensuring clarity before execution.

---



In [ ]:
import json, os, re
from groq_client import call_llm
from prompts import Few_Shot_Transformation

def extract_json(response_text):
    """
    Robust JSON extraction with regex fallback.
    Ensures you always get structured output.
    """
    try:
        return json.loads(response_text)
    except json.JSONDecodeError:
        try:
            match = re.search(r'\{.*\}', response_text, re.DOTALL)
            if match:
                return json.loads(match.group())
            return {"error": "No JSON found"}
        except Exception as e:
            return {"error": f"JSON extraction failed: {str(e)}"}

def save_output(filename, data):
    """
    Save parsed JSON only (no raw text file).
    """
    try:
        os.makedirs("outputs", exist_ok=True)

        with open(os.path.join("outputs", filename), "w") as f:
            json.dump(data, f, indent=2)

    except OSError as e:
        print(f"File operation error: {e}")
    except Exception as e:
        print(f"Unexpected error while saving output: {e}")

def run_case(case_name, prompt, temperature=0.2):
    """
    Call the LLM, parse response, save outputs, and return parsed JSON.
    """
    try:
        response = call_llm(prompt, temperature=temperature)
        print(f"RAW RESPONSE for {case_name}:\n{response}\n")

        parsed = extract_json(response)
        save_output(f"{case_name}.json", parsed)

        return parsed

    except ConnectionError as e:
        print(f"LLM connection error: {e}")
        return {"error": str(e)}

    except TimeoutError as e:
        print(f"LLM request timed out: {e}")
        return {"error": str(e)}

    except Exception as e:
        print(f"Unexpected error in run_case: {e}")
        return {"error": str(e)}


## Case 1.1

In [ ]:
# Case 1.1

from prompts import zero_shot_vendor_risk_prompt

result = run_case("1.1_zero_shot_vendor_risk_outputs", zero_shot_vendor_risk_prompt)
print("PARSED RESULT:", result)


RAW RESPONSE for zero_shot_vendor_risk_outputs:
```
{
  "risk_level": "MEDIUM",
  "key_risk_factors": [
    "Lack of region-specific data residency",
    "Insufficient SOC 2 certification (only Type I)",
    "Unclear rate limits for APIs",
    "Usage-based pricing with potential for significant increases",
    "Customer data used for product improvement unless opted out"
  ],
  "missing_information": [
    "Detailed information on data encryption methods",
    "Disaster recovery and business continuity plans",
    "Vendor's financial stability and growth prospects"
  ],
  "business_recommendation": "Proceed with onboarding DocuMind AI, but negotiate a contract that addresses the key risk factors, such as data residency, API rate limits, and pricing. Additionally, consider implementing additional security measures, such as encryption key management, and closely monitor the vendor's performance and certification status.",
  "confidence_score": 0.7
}
```

PARSED RESULT: {'risk_level': 'ME

## Case 1.2

In [ ]:
# Case 1.2

from prompts import zero_shot_exec_decision_prompt

result = run_case("1.2_zero_shot_exec_decision_outputs", zero_shot_exec_decision_prompt)
print("PARSED RESULT:", result)



RAW RESPONSE for zero_shot_exec_decision_outputs:
```json
{
  "decision": "APPROVE_WITH_CONDITIONS",
  "rationale": "The proposed GenAI chatbot can potentially reduce the ticket load and improve response times, addressing the growing customer support demand. However, it's crucial to address compliance concerns, implement AI governance policies, and mitigate potential job losses.",
  "financial_considerations": [
    "Estimated implementation cost: $250,000",
    "Ongoing monthly cost: $30,000",
    "Projected payback period: 12 months",
    "Potential cost savings from reduced ticket load: 35%"
  ],
  "operational_considerations": [
    "Integration with existing customer support systems",
    "Training and support for human agents to work with the chatbot",
    "Monitoring and evaluation of chatbot performance",
    "Change management for the support team"
  ],
  "people_impact": [
    "Potential job losses among human agents",
    "Need for retraining and upskilling for agents to wor

## Case 2.1

In [ ]:
# Case 2.1

from prompts import few_shot_ticket_prompt_new

result = run_case("2.1_few_shot_outputs_Vikash", few_shot_ticket_prompt_new)
print("PARSED RESULT:", result)



RAW RESPONSE for few_shot_outputs_Vikash:
[
  {
    "ticket": "I was charged twice this month and your support team has not replied for five days. I am going to post this on social media if this is not fixed today.",
    "category": "BILLING_ISSUE",
    "priority": "URGENT",
    "justification": "Double charge + delay + escalation risk + angry tone + public escalation threat"
  },
  {
    "ticket": "The export button stopped working after your latest update. Our reporting team is blocked.",
    "category": "TECHNICAL_BUG",
    "priority": "HIGH",
    "justification": "Critical feature broken + impact on business operations"
  },
  {
    "ticket": "Can you add approval workflows before invoices are submitted?",
    "category": "FEATURE_REQUEST",
    "priority": "MEDIUM",
    "justification": "New functionality request + no urgency or business impact mentioned"
  },
  {
    "ticket": "We need confirmation that our customer data is not being used to train your AI models.",
    "category":

## Case 2.2

In [ ]:
# Case 2.2

from prompts import few_shot_leave_prompt

result = run_case("2.2_few_shot_leave_outputs", few_shot_leave_prompt)
print("PARSED RESULT:", result)




RAW RESPONSE for few_shot_leave_outputs:
Here are the conversions of the user requests into structured API payloads:

1. User: "I want to take leave from 12th June to 15th June because I am travelling."
Output: 
{
  "action": "APPLY_LEAVE",
  "parameters": {"start_date": "2024-06-12", "end_date": "2024-06-15", "reason": "travelling"},
  "requires_clarification": false,
  "clarification_question": "",
  "confidence": 0.95
}

2. User: "How many casual leaves do I have left?"
Output: 
{
  "action": "CHECK_BALANCE",
  "parameters": {"leave_type": "casual"},
  "requires_clarification": false,
  "clarification_question": "",
  "confidence": 0.9
}

3. User: "Cancel my leave request for next Friday."
Output: 
{
  "action": "CANCEL_LEAVE",
  "parameters": {},
  "requires_clarification": true,
  "clarification_question": "Which exact date is 'next Friday'?",
  "confidence": 0.7
}

4. User: "What is the policy for maternity leave?"
Output: 
{
  "action": "GET_POLICY",
  "parameters": {"leave_type

## Case 3.1

In [ ]:
# Case 3.1

from prompts import chain_of_thought_roi_prompt

result = run_case("3.1_chain_of_thought_roi_outputs", chain_of_thought_roi_prompt)
print("PARSED RESULT:", result)


RAW RESPONSE for chain_of_thought_roi_outputs:
```json
{
  "incremental_revenue_range": "$80,000 - $140,000",
  "incremental_gross_profit_range": "$32,000 - $56,000",
  "monthly_net_benefit_range": "$2,000 - $26,000",
  "payback_period_range_months": "7 - 9 months",
  "decision": "APPROVE",
  "reasoning_summary": "The expected revenue uplift and gross profit increase justify the implementation cost, and the payback period is within the required 12 months after go-live.",
  "key_assumptions": [
    "Revenue uplift ranges from 4% to 7%",
    "Gross margin remains constant at 40%",
    "Implementation time is 3 months",
    "Monthly AI operating costs are $30,000"
  ]
}
```

PARSED RESULT: {'incremental_revenue_range': '$80,000 - $140,000', 'incremental_gross_profit_range': '$32,000 - $56,000', 'monthly_net_benefit_range': '$2,000 - $26,000', 'payback_period_range_months': '7 - 9 months', 'decision': 'APPROVE', 'reasoning_summary': 'The expected revenue uplift and gross profit increase ju

## Case 3.2

In [ ]:
# Case 3.2

from prompts import chain_of_thought_root_cause_prompt

result = run_case("3.2_chain_of_thought_root_cause_outputs", chain_of_thought_root_cause_prompt)
print("PARSED RESULT:", result)



RAW RESPONSE for chain_of_thought_root_cause_outputs:
```json
{
  "most_likely_causes": [
    "Concept drift due to changes in fraud patterns after the promotional campaign",
    "Data drift caused by the shift in feature distribution for transaction_amount"
  ],
  "evidence": [
    "Significant shift in transaction_amount feature distribution",
    "Introduction of a new payment channel",
    "Changes in fraud patterns after the promotional campaign",
    "Decrease in precision and F1-score"
  ],
  "less_likely_causes": [
    "Threshold miscalibration",
    "Pipeline failure"
  ],
  "recommended_diagnostics": [
    "Distribution checks for other features to identify potential data drift",
    "Retraining experiments with updated data to assess model adaptability",
    "Threshold tuning to evaluate its impact on model performance",
    "Analysis of fraud patterns to understand changes and update the model accordingly"
  ],
  "short_term_actions": [
    "Monitor model performance closel

## Case 4.1

In [ ]:
# Case 4.1

from prompts import llm_judge_support_prompt

result = run_case("4.1_llm_judge_support_outputs", llm_judge_support_prompt)
print("PARSED RESULT:", result)



RAW RESPONSE for llm_judge_support_outputs:
```
{
  "response_a": {
    "scores": {
      "empathy": 2,
      "helpfulness": 1,
      "clarity": 3
    },
    "strengths": [],
    "weaknesses": ["vague", "unhelpful", "dismissive"]
  },
  "response_b": {
    "scores": {
      "empathy": 5,
      "helpfulness": 5,
      "clarity": 5
    },
    "strengths": ["acknowledges frustration", "offers solution", "provides clear next steps"],
    "weaknesses": []
  },
  "winner": "B",
  "judge_reasoning_summary": "Response B is more empathetic, helpful, and clear. It acknowledges the customer's frustration, offers a solution by escalating the issue, and provides clear next steps. Response A is vague, unhelpful, and dismissive, making it an unsatisfactory response to the customer's concern."
}
```

PARSED RESULT: {'response_a': {'scores': {'empathy': 2, 'helpfulness': 1, 'clarity': 3}, 'strengths': [], 'weaknesses': ['vague', 'unhelpful', 'dismissive']}, 'response_b': {'scores': {'empathy': 5, 'help

## Case 4.2

In [ ]:
# Case 4.2

from prompts import llm_judge_code_explanation_prompt

result = run_case("4.2_llm_judge_code_explanation_outputs", llm_judge_code_explanation_prompt)
print("PARSED RESULT:", result)



RAW RESPONSE for llm_judge_code_explanation_outputs:
```json
{
  "explanation_a": {
    "scores": {
      "technical_accuracy": 9,
      "beginner_usefulness": 8
    },
    "issues": [
      "Lacks a clear explanation of the implications of shallow vs deep copying",
      "Does not provide examples or use cases"
    ],
    "overall_score": 8.5
  },
  "explanation_b": {
    "scores": {
      "technical_accuracy": 2,
      "beginner_usefulness": 4
    },
    "issues": [
      "Technically misleading claim that shallow copy is 'always bad'",
      "Oversimplification of memory allocation",
      "Fails to mention that deep copy can be slower and more memory-intensive",
      "Does not provide a clear understanding of when to use each type of copy"
    ],
    "overall_score": 3
  },
  "winner": "explanation_a",
  "judge_reasoning_summary": "Explanation A is more accurate and provides a clear distinction between shallow and deep copying. Explanation B makes misleading claims and oversimplif

## Case 5.1

In [ ]:
# Case 5.1

from prompts import self_consistency_policy_prompt

result = run_case("5.1_self_consistency_policy_outputs", self_consistency_policy_prompt)
print("PARSED RESULT:", result)



RAW RESPONSE for self_consistency_policy_outputs:
To evaluate the reimbursable amount, I will run multiple independent reasoning paths considering the given constraints and scenario.

**Private Reasoning Paths:**

1. **Path 1:** Calculate the daily meal limit for international travel: $60 * 1.25 = $75. Since the travel duration exceeds 8 hours but is a same-day trip, apply the 50% rule: $75 * 0.5 = $37.50. Subtract the alcohol expense: $70 - $12 = $58. However, this exceeds the calculated limit, so the reimbursable amount is $37.50.

2. **Path 2:** Determine if receipts are required: Since the claim is above $25, receipts are necessary. Given that receipts were provided, proceed with the calculation. For international same-day travel, the limit is $75 * 0.5 = $37.50. The alcohol expense is not reimbursable, but since the total claim minus alcohol ($58) still exceeds the limit, the reimbursable amount remains $37.50.

3. **Path 3:** Apply the same-day travel rule first: 50% of the stand

## Case 5.2

In [ ]:
# Case 5.2

from prompts import self_consistency_risk_prompt

result = run_case("5.2_self_consistency_risk_outputs", self_consistency_risk_prompt)
print("PARSED RESULT:", result)



RAW RESPONSE for self_consistency_risk_outputs:
```json
{
  "runs": [
    "Run 1: MEDIUM risk due to login outside business hours and MFA failure",
    "Run 2: MEDIUM risk due to login outside business hours and MFA failure",
    "Run 3: MEDIUM risk due to login outside business hours and MFA failure",
    "Run 4: MEDIUM risk due to login outside business hours and MFA failure",
    "Run 5: MEDIUM risk due to login outside business hours and MFA failure"
  ],
  "risk_level_votes": {
    "MEDIUM": 5,
    "HIGH": 0,
    "CRITICAL": 0,
    "LOW": 0
  },
  "final_risk_level": "MEDIUM",
  "disagreement_analysis": "No disagreement among runs, all runs classified the risk as MEDIUM",
  "final_reasoning_summary": "User Asha logged in from Germany, which is a known country and VPN country, outside business hours (8:15 PM) and failed MFA once. Although 8 files were downloaded, it does not meet the HIGH risk condition since Germany is not a new country. Therefore, the final risk level is MEDIUM."

## Case 6.1

In [ ]:
# Case 6.1

from prompts import tree_of_thought_use_case_prompt

result = run_case("6.1_tree_of_thought_use_case_outputs", tree_of_thought_use_case_prompt)
print("PARSED RESULT:", result)



RAW RESPONSE for tree_of_thought_use_case_outputs:
```json
{
  "options_evaluated": [
    {
      "option": "AI customer support assistant",
      "business_value_score": 4,
      "feasibility_score": 3,
      "risk_score": 2,
      "pilot_suitability_score": 3,
      "adoption_score": 3,
      "overall_score": 3,
      "trade_offs": ["High ticket volume", "Moderate implementation complexity", "Contains personal customer data"]
    },
    {
      "option": "AI sales proposal generator",
      "business_value_score": 4,
      "feasibility_score": 4,
      "risk_score": 4,
      "pilot_suitability_score": 4,
      "adoption_score": 5,
      "overall_score": 4,
      "trade_offs": ["Medium usage frequency", "Low data sensitivity", "Requires brand and legal review"]
    },
    {
      "option": "AI contract risk analyzer",
      "business_value_score": 5,
      "feasibility_score": 2,
      "risk_score": 1,
      "pilot_suitability_score": 2,
      "adoption_score": 3,
      "overall_score

## Case 6.2

In [ ]:
# Case 6.2

from prompts import tree_of_thought_architecture_prompt

result = run_case("6.2_tree_of_thought_architecture_outputs", tree_of_thought_architecture_prompt)
print("PARSED RESULT:", result)



RAW RESPONSE for tree_of_thought_architecture_outputs:
```json
{
  "architecture_scores": [
    {"Option A": 4, "rationale": "Balances accuracy and speed, suitable for MVP"},
    {"Option B": 2, "rationale": "Fine-tuning may not be feasible with frequent document changes"},
    {"Option C": 1, "rationale": "Lacks accuracy, may not meet user expectations"},
    {"Option D": 5, "rationale": "Highly accurate, but complex and may exceed MVP timeline"}
  ],
  "recommended_architecture": "Option A: Simple RAG with vector database and hosted LLM API",
  "implementation_rationale": "Option A provides a balance between accuracy and speed, is suitable for the MVP timeline, and respects the limited budget. It also allows for a phased approach, where more complex features can be added later.",
  "risks": [
    "Limited accuracy due to reliance on pre-trained LLM",
    "Potential scalability issues with growing user base",
    "Confidentiality concerns with hosted LLM API"
  ],
  "mitigations": [
 

## Case 7.1

In [ ]:
# Case 7.1

from prompts import ambiguous_request_rewrite_prompt

result = run_case("7.1_ambiguous_request_rewrite_outputs", ambiguous_request_rewrite_prompt)
print("PARSED RESULT:", result)



RAW RESPONSE for ambiguous_request_rewrite_outputs:
```
{
  "rephrased_problem": "Automate manual tasks in workflow management to reduce processing time by 30% and increase team output by 25% within the next 6 months, while providing real-time dashboards for leadership to track key performance indicators (KPIs) such as task completion rates, cycle times, and team workload.",
  "clarified_assumptions": [
    "Productivity is measured by the number of tasks completed per week and the time taken to complete each task.",
    "Visibility refers to the ability of leadership to access real-time data on team performance, task status, and potential bottlenecks in the workflow.",
    "Manual work refers to tasks that are currently performed manually, such as data entry, document review, and task assignment."
  ],
  "proposed_solution": "Implement an AI-powered workflow automation tool that uses machine learning algorithms to automate task assignment, predict task completion times, and identify p

## Case 7.2

In [ ]:
# Case 7.2

from prompts import poorly_written_requirement_prompt

result = run_case("7.2_poorly_written_requirement_outputs", poorly_written_requirement_prompt)
print("PARSED RESULT:", result)



RAW RESPONSE for poorly_written_requirement_outputs:
```
{
  "rephrased_requirement": "Design and implement a secure, efficient, and accurate AI-powered file analysis and question-answering system, where users can upload files, pose questions related to the content, and receive relevant and reliable responses within a specified timeframe.",
  "functional_requirements": [
    "Users can upload files in various formats (e.g., PDF, DOCX, TXT) up to a maximum size of 10MB",
    "The system can parse and analyze the uploaded files to extract relevant information",
    "Users can input questions related to the content of the uploaded files",
    "The system provides answers to user questions based on the analyzed file content",
    "The system supports a minimum of 100 concurrent user sessions"
  ],
  "non_functional_requirements": [
    "The system responds to user queries within 5 seconds (average response time) and 10 seconds (maximum response time) for 95% of the user interactions",
    

## Final Recommendations
- Use **Zero-Shot** for straightforward schema-driven tasks.  
- Apply **Few-Shot** when boundary conditions and ambiguity need to be taught.  
- Employ **Chain-of-Thought** for numerical or diagnostic reasoning.  
- Leverage **LLM-as-Judge** for evaluation tasks with clear rubrics.  
- Use **Self-Consistency** for high-stakes logic where reliability matters.  
- Apply **Tree-of-Thought** for multi-criteria decision-making.  
- Always consider **Rephrase-and-Respond** when requirements are vague or incomplete.

---

## Closing Note
Prompt engineering is not about memorizing definitions but about adapting techniques to context. Each method has trade-offs, and the most effective approach often combines multiple strategies for robustness and clarity.


## Created by - Vikash Kumar [AG0847]